# Stable NanoGPT-1GPU Recipe + SSONN MLP

This notebook combines the stable parts of `Deveraux-Parker/nanoGPT_1GPU_SPEEDRUN` with the SSONN experiment lifecycle.

It has two modes:

1. `RUN_MODE = "pretrain"`: train an **attention-only** GPT from scratch on FineWeb-Edu and save a checkpoint.
2. `RUN_MODE = "ssonn"`: load that checkpoint into the same GPT with a full SSONN MLP after each attention block. Attention weights are loaded and preserved across SSONN reinitialization; only the new SSONN MLPs are reset after expansion.

The original speedrun uses nightly PyTorch, FlexAttention, BF16, FP8, and an RTX 4090. This notebook intentionally keeps its reproducible training recipe but uses stable PyTorch 2.6 compatible operations: FP16 autocast + GradScaler and standard causal scaled-dot-product attention. Therefore it preserves the architecture scale, data format, optimizer idea, progressive context schedule, and update count, but does **not** claim the 90-minute/3.25-loss result on a TITAN RTX.

In [ ]:
# One additional dependency is needed for GPT-2 BPE tokenization:
# %pip install tiktoken

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import glob
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tqdm.auto import tqdm

import ssonn.train.train as ssonn_train
from ssonn.metrics.edge_finder import EdgeFinder
from ssonn.metrics.nonlinearity_metrics import AbsGradientEdgeMetric
from ssonn.model.model import ScaledNet

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Open the notebook from the repository root or experiments directory.")

DATA_DIR = REPO_ROOT / "data" / "finewebedu10B"
CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("This experiment is configured for CUDA training.")

# TITAN RTX has Tensor Cores but not BF16/FP8 training support required by the speedrun.
AMP_DTYPE = torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(torch.cuda.get_device_name(), "|", torch.cuda.get_device_properties(0).total_memory // 2**30, "GiB")

## Recipe Configuration

The model dimensions, data/tokenizer, Muon+Adam split, logit softcap, zero-initialized attention output projection, and `1750` optimizer updates are retained from the 4090 recipe.

The speedrun processes 32k-token local-attention sequences using FlexAttention. Standard PyTorch causal attention cannot safely run that on a 24GB TITAN RTX, so the stable version uses the same progressive attention window as its actual sequence length. Consequently it runs the same number of optimizer updates, but not the original 0.92B token budget.

In [ ]:
RUN_MODE = "pretrain"  # "pretrain" or "ssonn"
SEED = 1337
USE_COMPILE = False  # Stable PyTorch compile is optional; start with False on TITAN RTX.

# FineWeb-Edu data produced by the speedrun repository uses GPT-2 BPE uint16 shards.
# Both the speedrun downloader and local tokenizer use these wildcard patterns.
TRAIN_GLOB = str(DATA_DIR / "*train_*.bin")
VAL_GLOB = str(DATA_DIR / "*val_*.bin")

# Recommended: download the speedrun's already GPT-2-tokenized 100M-token shards.
DOWNLOAD_PRETOKENIZED_DATA = False
PRETOKENIZED_TRAIN_SHARDS = 10  # 10 * 100M = about 1B training tokens.

# Slower fallback: tokenize the raw FineWeb-Edu stream locally.
PREPARE_DATA = False
PREPARE_TRAIN_TOKENS = 1_000_000_000
PREPARE_VAL_TOKENS = 10_000_000

@dataclass(frozen=True)
class GPTConfig:
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 6
    n_embd: int = 768
    max_seq_len: int = 1792

MODEL_CONFIG = GPTConfig()

# Same update count, effective number of sequences/update, and context curriculum as the speedrun.
NUM_UPDATES = 1750
MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
WINDOW_START = 256
WINDOW_END = 1792
COOLDOWN_UPDATES = 640
EVAL_INTERVAL = 50
EVAL_BATCHES = 16
CHECKPOINT_INTERVAL = 250

# SSONN stage uses virtual epochs so the library expansion rule remains epoch-based.
SSONN_EPOCHS = 70
SSONN_STEPS_PER_EPOCH = 25
SSONN_EVAL_BATCHES = 16
FINDER_BATCHES = 32
SSONN_LEARNING_RATE = 2e-5
SSONN_T_MAX = 5
REINIT_MODE = "ssonn_only"  # Attention is never reset in this notebook.

BASE_CHECKPOINT = CHECKPOINT_DIR / "finewebedu_attention_only.pt"
SSONN_CHECKPOINT = CHECKPOINT_DIR / "finewebedu_ssonn_best.pt"

assert MODEL_CONFIG.n_embd % MODEL_CONFIG.n_head == 0
assert WINDOW_START <= WINDOW_END <= MODEL_CONFIG.max_seq_len
assert MICRO_BATCH_SIZE == 1, "The stable document stream uses one sequence per microbatch."
print("effective batch tokens at max window:", GRAD_ACCUM_STEPS * WINDOW_END)

In [ ]:
experiment = None
if os.environ.get("COMET_API_KEY") and os.environ.get("COMET_WORKSPACE"):
    from comet_ml import start

    experiment = start(
        api_key=os.environ["COMET_API_KEY"],
        workspace=os.environ["COMET_WORKSPACE"],
        project_name=os.environ.get("COMET_PROJECT_NAME", "ssonn-experiments"),
    )
    experiment.set_name(f"nanogpt_speedrun_ssonn_{RUN_MODE}_seed{SEED}")
    experiment.log_parameters({
        "run_mode": RUN_MODE,
        "model": asdict(MODEL_CONFIG),
        "num_updates": NUM_UPDATES,
        "micro_batch_size": MICRO_BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "window_start": WINDOW_START,
        "window_end": WINDOW_END,
        "cooldown_updates": COOLDOWN_UPDATES,
        "source_recipe": "Deveraux-Parker/nanoGPT_1GPU_SPEEDRUN@135bc07555d47b64223c03e3da0bc6571efc6f2a",
        "stable_deviations": "no FlexAttention, FP8, BF16, torchao, or custom Triton kernels",
        "gpu": torch.cuda.get_device_name(),
    })
else:
    print("Comet logging disabled: set COMET_API_KEY and COMET_WORKSPACE to enable it.")


## FineWeb-Edu Data

The recommended path downloads the pretokenized GPT-2 BPE shards published for the speedrun. It avoids network streaming, parquet parsing, and local BPE tokenization.

Set `DOWNLOAD_PRETOKENIZED_DATA = True` once. The fallback local tokenizer remains available as `PREPARE_DATA = True`, but writes each shard incrementally rather than holding one billion Python integers in memory.

In [ ]:
def write_datafile(path: Path, tokens: np.ndarray) -> None:
    header = np.zeros(256, dtype=np.int32)
    header[0] = 20240520
    header[1] = 1
    header[2] = len(tokens)
    with path.open("wb") as f:
        f.write(header.tobytes())
        f.write(tokens.astype(np.uint16, copy=False).tobytes())


def download_pretokenized_fineweb_edu(train_shards: int) -> None:
    from huggingface_hub import hf_hub_download

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    repo_id = "kjj0/finewebedu10B-gpt2"
    filenames = ["finewebedu_val_000000.bin"]
    filenames.extend(f"finewebedu_train_{index:06d}.bin" for index in range(1, train_shards + 1))
    for filename in filenames:
        print("Downloading", filename)
        hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=filename, local_dir=DATA_DIR)


def prepare_fineweb_edu(train_token_limit: int, val_token_limit: int, shard_size: int = 100_000_000) -> None:
    from array import array
    import tiktoken

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    enc = tiktoken.get_encoding("gpt2")
    eot = enc.eot_token
    dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
    rows = iter(dataset)

    def fill_buffer(limit: int) -> np.ndarray:
        buffer = array("H")
        for row in tqdm(rows, desc=f"Tokenizing {limit / 1e6:.0f}M tokens"):
            buffer.extend(enc.encode_ordinary(row["text"]))
            buffer.append(eot)
            if len(buffer) >= limit:
                return np.frombuffer(buffer, dtype=np.uint16, count=limit).copy()
        raise RuntimeError("FineWeb-Edu stream ended before reaching the requested token limit.")

    val_tokens = fill_buffer(val_token_limit)
    write_datafile(DATA_DIR / "fineweb_edu_val_000000.bin", val_tokens)

    written_tokens = 0
    shard_index = 0
    while written_tokens < train_token_limit:
        count = min(shard_size, train_token_limit - written_tokens)
        shard_tokens = fill_buffer(count)
        write_datafile(DATA_DIR / f"fineweb_edu_train_{shard_index:06d}.bin", shard_tokens)
        written_tokens += len(shard_tokens)
        shard_index += 1
        print(f"Wrote {written_tokens / 1e6:.0f}M / {train_token_limit / 1e6:.0f}M train tokens")


if DOWNLOAD_PRETOKENIZED_DATA:
    download_pretokenized_fineweb_edu(PRETOKENIZED_TRAIN_SHARDS)
elif PREPARE_DATA:
    prepare_fineweb_edu(PREPARE_TRAIN_TOKENS, PREPARE_VAL_TOKENS)

In [ ]:
def read_shard(path: str) -> np.ndarray:
    raw = np.memmap(path, dtype=np.uint16, mode="r")
    header = np.memmap(path, dtype=np.int32, mode="r", shape=(256,))
    if int(header[0]) != 20240520:
        raise ValueError(f"Unexpected FineWeb-Edu shard header in {path}")
    token_count = int(header[2])
    return raw[512 : 512 + token_count]


class TokenShardStream:
    def __init__(self, pattern: str, seed: int = 0):
        self.files = sorted(glob.glob(pattern))
        if not self.files:
            raise FileNotFoundError(f"No shards found for {pattern}. Run the preparation cell first.")
        self.rng = random.Random(seed)
        self.shard_index = 0
        self.position = 0
        self.tokens = read_shard(self.files[self.shard_index])

    def _advance(self) -> None:
        self.shard_index = (self.shard_index + 1) % len(self.files)
        self.position = 0
        self.tokens = read_shard(self.files[self.shard_index])

    def next_batch(self, seq_len: int) -> tuple[torch.Tensor, torch.Tensor]:
        required = seq_len + 1
        if self.position + required >= len(self.tokens):
            self._advance()
        block = torch.from_numpy(np.array(self.tokens[self.position : self.position + required], dtype=np.int64))
        self.position += seq_len
        return block[:-1].pin_memory(), block[1:].pin_memory()


class FixedFinderLoader:
    def __init__(self, pattern: str, batches: int, seq_len: int, seed: int):
        self.pattern = pattern
        self.batches = batches
        self.seq_len = seq_len
        self.seed = seed

    def __len__(self) -> int:
        return self.batches

    def __iter__(self):
        stream = TokenShardStream(self.pattern, self.seed)
        for _ in range(self.batches):
            x, y = stream.next_batch(self.seq_len)
            yield x.unsqueeze(0), y.unsqueeze(0)


def to_device(x: torch.Tensor, y: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    return x.unsqueeze(0).to(device, non_blocking=True), y.unsqueeze(0).to(device, non_blocking=True)

# Show the expected paths without stopping notebook setup before data is downloaded.
train_files = sorted(glob.glob(TRAIN_GLOB))
val_files = sorted(glob.glob(VAL_GLOB))
if train_files and val_files:
    train_stream = TokenShardStream(TRAIN_GLOB, seed=SEED)
    val_stream = TokenShardStream(VAL_GLOB, seed=SEED + 1)
    x0, y0 = train_stream.next_batch(WINDOW_START)
    print("data dir:", DATA_DIR)
    print("train shards:", len(train_files), "val shards:", len(val_files), "sample:", tuple(x0.shape), tuple(y0.shape))
else:
    print("No FineWeb-Edu shards found in", DATA_DIR)
    print("Set DOWNLOAD_PRETOKENIZED_DATA = True and run the previous cell once.")

## Stable Speedrun-Style Attention-Only GPT

Transferred from the speedrun: 12 layers, 6 heads, width 768, GPT-2 vocabulary, RoPE, RMSNorm/QK-norm, MQA parameterization, zero-initialized attention output projection, Muon for matrices, AdamW for embeddings/norm scalars, and tanh logit softcap.

Not transferred: FlexAttention/document block masks, FP8, BF16, `torchao`, and custom Triton kernels. Those require the nightly/Hopper environment explicitly excluded by this experiment.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, width: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.rms_norm(x, (x.size(-1),), self.weight, self.eps)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int, base: float = 10_000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, heads, seq, head_dim]
        seq_len = x.size(-2)
        cos = self.cos[:seq_len].to(dtype=x.dtype, device=x.device)[None, None]
        sin = self.sin[:seq_len].to(dtype=x.dtype, device=x.device)[None, None]
        x_even, x_odd = x[..., 0::2], x[..., 1::2]
        rotated = torch.stack((x_even * cos - x_odd * sin, x_even * sin + x_odd * cos), dim=-1)
        return rotated.flatten(-2)


class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.q_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.k_proj = nn.Linear(config.n_embd, self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, config.max_seq_len)
        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)

        nn.init.normal_(self.q_proj.weight, std=0.5 * config.n_embd ** -0.5)
        nn.init.normal_(self.k_proj.weight, std=0.5 * config.n_embd ** -0.5)
        nn.init.normal_(self.v_proj.weight, std=0.5 * config.n_embd ** -0.5)
        nn.init.zeros_(self.o_proj.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch, seq_len, 1, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch, seq_len, 1, self.head_dim).transpose(1, 2)
        q = self.rope(self.q_norm(q))
        k = self.rope(self.k_norm(k))
        # Expand one KV head instead of requiring nightly FlexAttention/GQA kernels.
        k = k.expand(-1, self.n_head, -1, -1)
        v = v.expand(-1, self.n_head, -1, -1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(y.transpose(1, 2).contiguous().view(batch, seq_len, -1))


class TokenSSONNMLP(nn.Module):
    def __init__(self, width: int, device_name: str):
        super().__init__()
        self.scaled = ScaledNet(torch.eye(width), torch.zeros(width), device=device_name)
        self.scaled.dropout.p = 0.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, width = x.shape
        return self.scaled(x.reshape(batch * seq_len, width)).reshape(batch, seq_len, width)


class TransformerBlock(nn.Module):
    def __init__(self, config: GPTConfig, use_ssonn_mlp: bool = False):
        super().__init__()
        self.norm_attn = RMSNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.norm_mlp = RMSNorm(config.n_embd)
        self.mlp = TokenSSONNMLP(config.n_embd, str(device)) if use_ssonn_mlp else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm_attn(x))
        if self.mlp is not None:
            # Full adapter, matching the prior SSONN experiment rather than a residual delta.
            x = self.mlp(self.norm_mlp(x))
        return x


class AttentionOnlyGPT(nn.Module):
    def __init__(self, config: GPTConfig, use_ssonn_mlp: bool = False):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.blocks = nn.ModuleList([TransformerBlock(config, use_ssonn_mlp) for _ in range(config.n_layer)])
        self.norm_final = RMSNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.ssonn_layer_names: list[str] = []
        if use_ssonn_mlp:
            for idx, block in enumerate(self.blocks):
                name = f"ssonn_mlp_{idx}"
                setattr(self, name, block.mlp.scaled)
                self.ssonn_layer_names.append(name)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        x = self.wte(tokens)
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.norm_final(x))
        return 30.0 * torch.tanh(logits / 30.0)


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())


model = AttentionOnlyGPT(MODEL_CONFIG, use_ssonn_mlp=RUN_MODE == "ssonn").to(device)
if USE_COMPILE:
    model = torch.compile(model)
print(f"parameters: {count_parameters(model) / 1e6:.1f}M | ssonn layers: {len(model.ssonn_layer_names)}")

## Checkpoint Interface

The base checkpoint stores only attention-only weights. In SSONN mode the checkpoint is loaded with `strict=False`; missing keys must be exactly the newly introduced SSONN MLPs. This guarantees that downstream experiments begin from the same attention backbone.

In [ ]:
def checkpoint_payload(model: nn.Module, optimizer_state: dict | None, update: int) -> dict:
    return {
        "model": model.state_dict(),
        "model_config": asdict(MODEL_CONFIG),
        "update": update,
        "optimizer": optimizer_state,
        "recipe": {
            "source": "Deveraux-Parker/nanoGPT_1GPU_SPEEDRUN@135bc07555d47b64223c03e3da0bc6571efc6f2a",
            "stable_deviations": ["no FlexAttention", "no FP8", "no BF16", "no torchao", "standard SDPA"],
            "num_updates": NUM_UPDATES,
            "window_start": WINDOW_START,
            "window_end": WINDOW_END,
            "effective_sequences_per_update": GRAD_ACCUM_STEPS,
        },
        "rng": {
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all(),
            "numpy": np.random.get_state(),
            "python": random.getstate(),
        },
    }


def load_attention_checkpoint(model: AttentionOnlyGPT, path: Path) -> dict:
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    missing, unexpected = model.load_state_dict(checkpoint["model"], strict=False)
    if unexpected:
        raise RuntimeError(f"Unexpected checkpoint keys: {unexpected}")
    if not all("mlp" in name or "ssonn_mlp" in name for name in missing):
        raise RuntimeError(f"Checkpoint is incompatible with this backbone. Missing: {missing}")
    return checkpoint


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## Speedrun-Inspired Pretraining

Matrix parameters use Muon; embeddings and norm scales use AdamW. This is the stable counterpart to the source's Muon/Adam split. The window schedule and cooldown are update-based, exactly as in the source recipe. `GRAD_ACCUM_STEPS=16` preserves its effective sequence count while keeping a 24GB TITAN RTX safe with standard attention.

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, stream: TokenShardStream, seq_len: int, batches: int) -> float:
    model.eval()
    losses = []
    for _ in range(batches):
        x_cpu, y_cpu = stream.next_batch(seq_len)
        x, y = to_device(x_cpu, y_cpu)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        losses.append(loss.item())
    return float(np.mean(losses))


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr: float = 0.02, momentum: float = 0.95, nesterov: bool = True, steps: int = 5):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov, steps=steps))

    @staticmethod
    def orthogonalize(gradient: torch.Tensor, steps: int) -> torch.Tensor:
        x = gradient.float()
        x = x / (x.norm() + 1e-7)
        transposed = x.size(0) > x.size(1)
        if transposed:
            x = x.t()
        a, b, c = 3.4445, -4.7750, 2.0315
        for _ in range(steps):
            a_mat = x @ x.t()
            x = a * x + (b * a_mat + c * a_mat @ a_mat) @ x
        if transposed:
            x = x.t()
        return x.to(dtype=gradient.dtype)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for parameter in group["params"]:
                if parameter.grad is None:
                    continue
                state = self.state[parameter]
                if "momentum" not in state:
                    state["momentum"] = torch.zeros_like(parameter.grad)
                momentum = state["momentum"]
                momentum.mul_(group["momentum"]).add_(parameter.grad)
                update = parameter.grad.add(momentum, alpha=group["momentum"]) if group["nesterov"] else momentum
                update = self.orthogonalize(update, group["steps"])
                update.mul_(max(1.0, update.size(0) / update.size(1)) ** 0.5)
                parameter.add_(update, alpha=-group["lr"])


def make_pretrain_optimizers(model: nn.Module):
    matrix, adam = [], []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if parameter.ndim == 2 and name not in {"wte.weight", "lm_head.weight"}:
            matrix.append(parameter)
        else:
            adam.append(parameter)
    return Muon(matrix, lr=0.05, momentum=0.85), torch.optim.AdamW(adam, lr=6e-4, betas=(0.8, 0.95), weight_decay=0.0)


def speedrun_window(step: int) -> int:
    fraction = step / max(NUM_UPDATES - 1, 1)
    raw = WINDOW_START + fraction * (WINDOW_END - WINDOW_START)
    return min(WINDOW_END, max(WINDOW_START, int(raw // 256) * 256))


def speedrun_lr_scale(step: int) -> float:
    cooldown_start = NUM_UPDATES - COOLDOWN_UPDATES
    return 1.0 if step < cooldown_start else (NUM_UPDATES - step) / COOLDOWN_UPDATES


def pretrain_attention_only(model: AttentionOnlyGPT) -> AttentionOnlyGPT:
    if RUN_MODE != "pretrain":
        raise ValueError("Set RUN_MODE='pretrain' for this cell.")
    model.train()
    muon, adam = make_pretrain_optimizers(model)
    scaler = torch.amp.GradScaler("cuda")
    train_stream = TokenShardStream(TRAIN_GLOB, seed=SEED)
    val_stream = TokenShardStream(VAL_GLOB, seed=SEED + 1)
    started = time.perf_counter()
    tokens_seen = 0

    for step in range(NUM_UPDATES):
        seq_len = speedrun_window(step)
        muon.param_groups[0]["momentum"] = 0.85 + 0.10 * min(step / 300, 1.0)
        lr_scale = speedrun_lr_scale(step)
        muon.param_groups[0]["lr"] = 0.05 * lr_scale
        adam.param_groups[0]["lr"] = 6e-4 * lr_scale
        muon.zero_grad(set_to_none=True)
        adam.zero_grad(set_to_none=True)

        loss_sum = 0.0
        for _ in range(GRAD_ACCUM_STEPS):
            x_cpu, y_cpu = train_stream.next_batch(seq_len)
            x, y = to_device(x_cpu, y_cpu)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten()) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            # loss is already divided by GRAD_ACCUM_STEPS; summing it gives the mean raw loss.
            loss_sum += loss.detach().item()

        tokens_seen += GRAD_ACCUM_STEPS * seq_len
        scaler.unscale_(muon)
        scaler.unscale_(adam)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(muon)
        scaler.step(adam)
        scaler.update()

        if (step + 1) % EVAL_INTERVAL == 0 or step == 0:
            val_loss = evaluate(model, val_stream, seq_len, EVAL_BATCHES)
            elapsed = time.perf_counter() - started
            tokens_per_second = tokens_seen / max(elapsed, 1)
            print(f"update {step + 1:4d}/{NUM_UPDATES} | window {seq_len:4d} | train {loss_sum:.4f} | val {val_loss:.4f} | {tokens_per_second:.0f} tok/s")
            if experiment is not None:
                experiment.log_metrics({
                    "pretrain/train_loss": loss_sum,
                    "pretrain/val_loss": val_loss,
                    "pretrain/val_perplexity": math.exp(min(val_loss, 20.0)),
                    "pretrain/window": seq_len,
                    "pretrain/tokens_seen": tokens_seen,
                    "pretrain/tokens_per_second": tokens_per_second,
                }, step=step + 1)
            model.train()

        if (step + 1) % CHECKPOINT_INTERVAL == 0 or step + 1 == NUM_UPDATES:
            torch.save(checkpoint_payload(model, {"muon": muon.state_dict(), "adam": adam.state_dict()}, step + 1), BASE_CHECKPOINT)
            print("saved", BASE_CHECKPOINT)
    return model


# Run this only after FineWeb-Edu shards exist.
# pretrained_model = pretrain_attention_only(model)

## SSONN Full-MLP Stage

Switch to `RUN_MODE = "ssonn"`, rerun model construction, then run this stage. The attention-only checkpoint is loaded into the new model with SSONN MLPs. At expansion, only `ScaledNet` modules are reinitialized. Attention is trainable, but never reset.

In [ ]:
def ssonn_only_weights_init(module: nn.Module) -> None:
    if isinstance(module, ScaledNet):
        module.reinitialize_parameters()


def lm_cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    # EdgeFinder calls metric.calculate(...), which applies squeeze() to a batch of one.
    # Support both regular [batch, seq, vocab] logits and squeezed [seq, vocab] logits.
    if logits.ndim == 3:
        logits = logits.flatten(0, 1)
    elif logits.ndim != 2:
        raise ValueError(f"Expected [batch, seq, vocab] or [seq, vocab], got {tuple(logits.shape)}")
    return F.cross_entropy(logits, targets.flatten())


def train_ssonn_stage(model: AttentionOnlyGPT) -> AttentionOnlyGPT:
    if RUN_MODE != "ssonn":
        raise ValueError("Set RUN_MODE='ssonn' and rerun model construction before this cell.")
    if not BASE_CHECKPOINT.exists():
        raise FileNotFoundError(f"Missing base checkpoint: {BASE_CHECKPOINT}")
    load_attention_checkpoint(model, BASE_CHECKPOINT)
    ssonn_train.weights_init = ssonn_only_weights_init

    optimizer = optim.AdamW(model.parameters(), lr=SSONN_LEARNING_RATE, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SSONN_T_MAX, eta_min=0.0)
    finder_loader = FixedFinderLoader(TRAIN_GLOB, FINDER_BATCHES, WINDOW_END, SEED + 2)
    edge_finder = EdgeFinder(
        metric=AbsGradientEdgeMetric(lm_cross_entropy),
        dataloader=finder_loader,
        device=device,
        aggregation_mode="mean",
        max_to_replace=1024,
    )
    thresholds = {name: 0.1 for name in model.ssonn_layer_names}
    train_stream = TokenShardStream(TRAIN_GLOB, SEED + 3)
    val_stream = TokenShardStream(VAL_GLOB, SEED + 4)
    best_val = float("inf")

    for epoch in range(SSONN_EPOCHS):
        model.train()
        train_losses = []
        for _ in tqdm(range(SSONN_STEPS_PER_EPOCH), desc=f"SSONN epoch {epoch + 1}", leave=False):
            x_cpu, y_cpu = train_stream.next_batch(WINDOW_END)
            x, y = to_device(x_cpu, y_cpu)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                loss = lm_cross_entropy(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        val_loss = evaluate(model, val_stream, WINDOW_END, SSONN_EVAL_BATCHES)
        expanded = False
        new_edges = 0
        if ((epoch - SSONN_T_MAX) % (2 * SSONN_T_MAX)) == 0 and epoch != 0:
            expanded, new_edges = ssonn_train.expand_model(model, edge_finder, thresholds)
            if expanded:
                model.apply(ssonn_train.weights_init)
                optimizer = optim.AdamW(model.parameters(), lr=optimizer.param_groups[0]["lr"], weight_decay=0.0)
                scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SSONN_T_MAX, eta_min=0.0, last_epoch=epoch)

        scheduler.step()
        total_params = count_parameters(model)
        if val_loss >= 20.0:
            ppl_text = ">= exp(20)"
        else:
            ppl_text = f"{math.exp(val_loss):.2f}"
        print(f"epoch {epoch + 1:02d}/{SSONN_EPOCHS} | train {np.mean(train_losses):.4f} | val {val_loss:.4f} | ppl {ppl_text} | expanded {expanded} edges {new_edges} | params {total_params / 1e6:.1f}M")
        if experiment is not None:
            experiment.log_metrics({
                "ssonn/train_loss": float(np.mean(train_losses)),
                "ssonn/val_loss": val_loss,
                "ssonn/val_perplexity": math.exp(min(val_loss, 20.0)),
                "ssonn/lr": optimizer.param_groups[0]["lr"],
                "ssonn/expanded": int(expanded),
                "ssonn/new_edges": int(new_edges),
                "ssonn/total_params": total_params,
            }, step=epoch + 1)
        if val_loss < best_val:
            best_val = val_loss
            torch.save(checkpoint_payload(model, optimizer.state_dict(), epoch + 1), SSONN_CHECKPOINT)
    return model


# ssonn_model = train_ssonn_stage(model)